# Fase 1 — Híbrido (Local / Colab)
**Auto-detecta** si se ejecuta en Google Colab o en PC local.

- En **Colab**: monta Drive y usa rutas de Drive
- En **Local**: usa rutas del sistema de archivos local

**ROI**: Shapefile (`Roigeneral.zip`)
**Bandas**: B2 (Blue), B3 (Green), B4 (Red) — Color natural

In [ ]:
# =============================================================================
# CELDA 1: AUTO-DETECCIÓN DE ENTORNO + VERIFICACIÓN
# Funciona en Colab y en PC local
# =============================================================================
import sys, subprocess, os, importlib

EN_COLAB = "google.colab" in sys.modules

LIBRERIAS = ["pystac_client", "geopandas", "rioxarray", "rasterio",
            "odc", "stackstac", "xarray", "matplotlib"]
LIBRERIAS_FALTANTES = [l for l in LIBRERIAS if not importlib.util.find_spec(l.split('.')[0])]

if EN_COLAB:
    print("Entorno: GOOGLE COLAB")
    if LIBRERIAS_FALTANTES:
        !pip install pystac-client stackstac rioxarray geopandas rasterio odc-stac -q
else:
    print("Entorno: PC LOCAL")
    if LIBRERIAS_FALTANTES:
        print(f"Faltan: {LIBRERIAS_FALTANTES}")
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + LIBRERIAS_FALTANTES + ["-q"])
    else:
        print("Todas las librerias ya instaladas.")

print("Entorno listo.")

In [ ]:
# =============================================================================
# CELDA 2: IMPORTACIONES
# =============================================================================
import os
import glob
import calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import rasterio
import matplotlib
matplotlib.use('Agg')  # Sin interfaz grafica (funciona en ambos entornos)
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from collections import Counter
from pystac_client import Client
import stackstac
from odc.stac import load
from datetime import datetime

if EN_COLAB:
    from google.colab import drive

print("✅ Importaciones completadas.")

In [ ]:
# =============================================================================
# CELDA 3: CONFIGURACIÓN UNIFICADA
# =============================================================================

if EN_COLAB:
    drive.mount('/content/drive')
    SHAPEFILE_PATH = "/content/drive/MyDrive/Tesis/GEE Murcott/ROI/4ParcelasDefinidas.zip"
    # En Colab: exporta al lado del shapefile
    BASE_DIR = os.path.join(os.path.dirname(SHAPEFILE_PATH), "output", "rgb", "4ParcelasDefinidas")
else:
    # Local: el shapefile en la carpeta del proyecto
    SCRIPT_DIR = os.getcwd()
    SHAPEFILE_PATH = os.path.join(SCRIPT_DIR, "4ParcelasDefinidas.zip")
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = "./4ParcelasDefinidas.zip"
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = input("Ruta del shapefile: ").strip()
    # En local: exporta dentro del proyecto
    BASE_DIR = os.path.join(SCRIPT_DIR, "output", "rgb", "4ParcelasDefinidas")

# ⚙️ PARÁMETROS DE BÚSQUEDA (configura libremente)
PERIODO_INICIO = "2025-01-01"
PERIODO_FIN = "2026-06-03"
# Sin filtro de nubes: se etiqueta cada escena como Despejada/Nublada/SinDatos

# ─── Calcular fechas por mes ────────────────────────────────────────────────
MES_NOMBRE = ["", "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
               "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

def generar_fechas_por_mes(inicio_str, fin_str):
    inicio = datetime.strptime(inicio_str, '%Y-%m-%d')
    fin = datetime.strptime(fin_str, '%Y-%m-%d')
    fechas = []
    corriente = inicio.replace(day=1)
    while corriente <= fin:
        a, m = corriente.year, corriente.month
        ultimo = calendar.monthrange(a, m)[1]
        fechas.append((
            corriente.strftime('%Y-%m-%d'),
            min(datetime.strptime(f'{a}-{m:02d}-{ultimo}', '%Y-%m-%d'), fin).strftime('%Y-%m-%d'),
            f'{m:02d} - {MES_NOMBRE[m]} {a}'
        ))
        corriente = datetime(a + (1 if m == 12 else 0), 1 if m == 12 else m + 1, 1)
    return fechas

FECHAS = generar_fechas_por_mes(PERIODO_INICIO, PERIODO_FIN)

print("✅ Configuración cargada.")
print(f"   Shapefile: {SHAPEFILE_PATH}")
print(f"   Período: {FECHAS[0][2] if FECHAS else 'N/A'}")
print(f"   Exportar a: {BASE_DIR}")

In [ ]:
# =============================================================================
# CELDA 4: FUNCIONES AUXILIARES
# =============================================================================

def tif_to_png_batch(input_dir, output_dir, cmap_name='viridis', percentiles=(2, 98)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 Carpeta creada: {output_dir}")
    files = sorted(glob.glob(os.path.join(input_dir, "*.tif")))
    print(f"🚀 Procesando {len(files)} archivos...")
    for tif_path in files:
        filename = os.path.basename(tif_path).replace('.tif', '.png')
        save_path = os.path.join(output_dir, filename)
        with rasterio.open(tif_path) as src:
            data = src.read(1).astype(np.float32)
            nodata = src.nodata
            if nodata is not None:
                data[data == nodata] = np.nan
        vmin, vmax = np.nanpercentile(data, percentiles)
        norm = Normalize(vmin=vmin, vmax=vmax, clip=True)
        plt.figure(figsize=(10, 10))
        plt.imshow(data, cmap=cmap_name, norm=norm)
        plt.axis('off')
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
        plt.close()
        print(f"✅ Convertido: {filename} [Paleta: {cmap_name}]")


def aplicar_mascara_scl(scl, clases_validas=[4,5]):
    """Mascara SCL: True donde la clase es valida"""
    mask = xr.zeros_like(scl, dtype=bool)
    for c in clases_validas:
        mask = mask | (scl == c)
    return mask

def aplicar_mascara_geometrica(da, gdf):
    """Mascara: solo pixeles DENTRO de las geometrias"""
    from rasterio.features import geometry_mask
    x_name = next((n for n in ['x', 'lon', 'longitude'] if n in da.coords), None)
    y_name = next((n for n in ['y', 'lat', 'latitude'] if n in da.coords), None)
    if not x_name or not y_name:
        return da
    transform = da.rio.transform()
    ny = da.sizes[y_name]
    nx = da.sizes[x_name]
    mascara = ~geometry_mask(gdf.geometry.values, transform=transform, out_shape=(ny, nx))
    mascara_da = xr.DataArray(mascara, dims=(y_name, x_name),
                               coords={y_name: da[y_name], x_name: da[x_name]})
    return da.where(mascara_da)

print("✅ Funciones auxiliares cargadas.")

In [ ]:
# =============================================================================
# CELDA 5: CARGAR SHAPEFILE + CALCULAR BBOX
# =============================================================================

print("🔍 Cargando shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"✅ Shapefile cargado: {len(gdf)} feature(s)")

if gdf.crs and gdf.crs.is_geographic:
    gdf_geo = gdf
else:
    gdf_geo = gdf.to_crs("EPSG:4326")

bbox = gdf_geo.total_bounds
print(f"   Bounding Box: {bbox}")

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
gdf_geo.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=1.5)
ax.set_title("ROI — Mandarina Murcott", fontsize=12)
plt.tight_layout()
plt.show()

# Directorio base de salida
os.makedirs(BASE_DIR, exist_ok=True)
print(f"✅ Directorio base: {BASE_DIR}")
print(f"   Meses a procesar: {len(FECHAS)}")

In [ ]:
# =============================================================================
# CELDA 6: BÚSQUEDA STAC + CONTEO + CLASIFICACIÓN DE NUBES
# =============================================================================

print("🔍 Conectando al catálogo Earth Search (AWS)...")
catalog = Client.open("https://earth-search.aws.element84.com/v1")
print("✅ Conexión exitosa.")

resultados_totales = []
for fecha_inicio, fecha_fin, label in FECHAS:
    print(f"\n📅 {label} ({fecha_inicio} → {fecha_fin}):")
    params = dict(
        collections=["sentinel-2-l2a"],
        bbox=list(bbox),
        datetime=f"{fecha_inicio}/{fecha_fin}",
    )
    if CLOUD_FILTER:
        params["query"] = CLOUD_FILTER
    search = catalog.search(**params)
    items = list(search.items())
    print(f"   Imágenes encontradas: {len(items)}")

    for item in items:
        cloud = item.properties.get("eo:cloud_cover", None)
        if cloud is None:
            estado = "Sin dato"; cloud_val = -1
        elif cloud < 10:
            estado = "Despejada"; cloud_val = cloud
        else:
            estado = "Nublada"; cloud_val = cloud
        resultados_totales.append({
            "fecha": item.datetime.strftime("%Y-%m-%d"),
            "id": item.id, "nubes_%": cloud_val,
            "estado": estado, "mes": label,
        })

df_resultados = pd.DataFrame(resultados_totales)
if len(df_resultados) == 0:
    print("\n❌ No se encontraron imágenes.")
else:
    print("\n" + "=" * 50)
    for (mes, fecha), group in df_resultados.groupby(["mes", "fecha"]):
        nubes_prom = group["nubes_%"].mean()
        if nubes_prom < 0:
            icon = "❓"
            estado_txt = "Sin dato"
        elif nubes_prom < 20:
            icon = "🌤️"
            estado_txt = f"Despejada ({nubes_prom:.0f}%)"
        elif nubes_prom < 60:
            icon = "⛅"
            estado_txt = f"Parcial ({nubes_prom:.0f}%)"
        else:
            icon = "☁️"
            estado_txt = f"Nublada ({nubes_prom:.0f}%)"
        print(f"  {fecha}: {len(group)} img | {icon} {estado_txt}")
    print(f"\n📈 TOTAL: {len(df_resultados)} escenas.")

In [ ]:
# =============================================================================
# CELDA 7: CARGA DEL DATA CUBE
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay imágenes para cargar.")
else:
    print("📦 Cargando Data Cube...")
    items_list = []
    for fecha_inicio, fecha_fin, label in FECHAS:
        params = dict(
            collections=["sentinel-2-l2a"],
            bbox=list(bbox),
            datetime=f"{fecha_inicio}/{fecha_fin}",
        )
        if CLOUD_FILTER:
            params["query"] = CLOUD_FILTER
        search = catalog.search(**params)
        items_list.extend(list(search.items()))
    print(f"   Total escenas: {len(items_list)}")

    ds = load(
        items_list,
        bands=["blue", "green", "red", "scl"],
        bbox=list(bbox),
        crs="EPSG:4326",
        resolution=0.0001,
        groupby=None,
        chunks={'time': 1, 'x': 512, 'y': 512},
    )
    print(f"✅ Data Cube: {dict(ds.sizes)}")

In [ ]:
# =============================================================================
# CELDA 8: VISUALIZACIÓN RGB
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos.")
else:
    print("🖼️ Generando visualización RGB...")
    red_band = ds["red"].values / 10000.0
    green_band = ds["green"].values / 10000.0
    blue_band = ds["blue"].values / 10000.0

    n_times = ds.sizes["time"]
    n_cols = min(4, n_times)
    n_rows = max(1, (n_times + n_cols - 1) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = [axes] if n_times == 1 else axes.flatten()

    for t in range(n_times):
        ax = axes[t]
        rgb = np.stack([red_band[t], green_band[t], blue_band[t]], axis=-1)
        rgb = np.clip(rgb, 0, 1)
        ax.imshow(rgb)
        ax.axis('off')
        ts = str(ds.time.values[t])[:16]
        ax.set_title(ts, fontsize=9)

    for t in range(n_times, len(axes)):
        axes[t].axis('off')
    plt.suptitle("RGB — Mandarina Murcott", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/PNG/Grid_RGB.png", dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✅ Grid guardado.")

In [ ]:
# =============================================================================
# CELDA 9: EXPORTAR PNG + GeoTIFF POR MES
# =============================================================================

from rasterio.features import geometry_mask

if len(df_resultados) == 0:
    print("❌ No hay datos para exportar.")
else:
    total_exportados = 0
    contador_global = 0
    for label, grupo in df_resultados.groupby('mes'):
        # Buscar las fechas para este mes
        fechas_mes = [f for f in FECHAS if f[2] == label]
        if not fechas_mes:
            continue
        fi, ff, _ = fechas_mes[0]
        print(f"\n📅 {label}:")

        # Obtener indices de tiempo para este mes
        mask_temporal = np.array([
            (pd.Timestamp(ts) >= pd.Timestamp(fi)) and (pd.Timestamp(ts) <= pd.Timestamp(ff))
            for ts in ds.time.values
        ])
        indices_mes = np.where(mask_temporal)[0]
        if len(indices_mes) == 0:
            print("   Sin escenas en este mes.")
            continue

        # Carpetas del mes
        mes_carpeta = os.path.join(BASE_DIR, label)
        png_dir = os.path.join(mes_carpeta, 'PNG')
        tif_dir = os.path.join(mes_carpeta, 'GeoTIFF')
        os.makedirs(png_dir, exist_ok=True)
        os.makedirs(tif_dir, exist_ok=True)

        contador = 0
        registros_mes = []

        for t in indices_mes:
            contador += 1
            contador_global += 1
            ts_val = ds.time.values[t]
            fecha_dt = pd.Timestamp(ts_val).to_pydatetime()
            fecha_str = fecha_dt.strftime('%Y%m%d')
            hora_str = fecha_dt.strftime('%H%M%S')
            escena = ds.isel(time=t)

            # Obtener cloud_cover de la fila correspondiente
            cloud_val = -1
            for _, row in grupo.iterrows():
                if row['fecha'] == fecha_dt.strftime('%Y-%m-%d'):
                    cloud_val = row['nubes_%']
                    break

            # Aplicar mascara SCL (clases 4,5) igual que NDRE
            if 'scl' in escena:
                mascara_scl = aplicar_mascara_scl(escena['scl'])
            else:
                mascara_scl = xr.ones_like(escena['red'], dtype=bool)

            # Escalar RGB y enmascarar con SCL
            r = escena['red'].where(mascara_scl) / 10000.0
            g = escena['green'].where(mascara_scl) / 10000.0
            b = escena['blue'].where(mascara_scl) / 10000.0

            # Aplicar mascara geometrica (recortar a parcelas)
            r = aplicar_mascara_geometrica(r, gdf_geo)
            g = aplicar_mascara_geometrica(g, gdf_geo)
            b = aplicar_mascara_geometrica(b, gdf_geo)

            data_values = r.values if hasattr(r, 'values') else r

            # Determinar sufijo
            if np.all(np.isnan(data_values)):
                sufijo = 'SinDatos'
            elif cloud_val < 10:
                sufijo = 'Despejada'
            else:
                sufijo = 'Nublada'

            mes_corto = label.split(' - ')[1].split()[0] if ' - ' in label else label
            nombre_base = f'RGB_{mes_corto}_{contador:03d}_{fecha_str}_{hora_str}_{sufijo}'

            # Exportar PNG
            rgb_np = np.stack([
                np.nan_to_num(r.values, nan=0),
                np.nan_to_num(g.values, nan=0),
                np.nan_to_num(b.values, nan=0),
            ], axis=-1)
            rgb_np = np.clip(rgb_np, 0, 1)
            rgb_uint8 = (rgb_np * 255).astype(np.uint8)
            if sufijo == 'SinDatos':
                rgb_uint8[:] = 240  # gris claro

            png_path = os.path.join(png_dir, f'{nombre_base}.png')
            plt.figure(figsize=(10, 10))
            plt.imshow(rgb_uint8)
            plt.axis('off')
            plt.savefig(png_path, bbox_inches='tight', pad_inches=0, dpi=200)
            plt.close()

            # Exportar GeoTIFF
            tif_path = os.path.join(tif_dir, f'{nombre_base}.tif')
            try:
                rgb_da = escena[["red", "green", "blue"]].to_array(dim="band")
                rgb_da.rio.to_raster(tif_path)
            except:
                pass

            registros_mes.append({
                'fecha': fecha_dt.strftime('%Y-%m-%d'),
                'hora': fecha_dt.strftime('%H:%M:%S'),
                'id_escena': nombre_base,
                'nubes_porciento': cloud_val,
                'estado_nubosidad': sufijo,
                'ruta_png': png_path,
                'ruta_tif': tif_path,
                'mes_label': label,
            }),
            total_exportados += 1

        # CSV de metadatos del mes
        if registros_mes:
            df_mes = pd.DataFrame(registros_mes)
            csv_path = os.path.join(mes_carpeta, 'metadatos.csv')
            df_mes.to_csv(csv_path, index=False, encoding='utf-8')

        despejadas = sum(1 for r in registros_mes if r['estado_nubosidad'] == 'Despejada')
        nubladas = sum(1 for r in registros_mes if r['estado_nubosidad'] == 'Nublada')
        sindatos = sum(1 for r in registros_mes if r['estado_nubosidad'] == 'SinDatos')
        print(f"   Exportado: {len(registros_mes)} PNGs + GeoTIFFs ({despejadas}D/{nubladas}N/{sindatos}S)")

    print(f"\n📁 Total exportados: {total_exportados} archivos en {BASE_DIR}")

In [ ]:
# =============================================================================
# CELDA 10: (OBSOLETO - La exportacion se hace en Celda 9)
# =============================================================================
print("✅ Exportacion ya completada en Celda 9.")

In [ ]:
# =============================================================================
# CELDA 11: REPORTE DE EXPORTACION
# =============================================================================

print("=" * 60)
print("   REPORTE FINAL - RGB")
print(f"   Periodo: {PERIODO_INICIO} a {PERIODO_FIN}")
print(f"   Directorio: {BASE_DIR}")
print("=" * 60)
if len(df_resultados) > 0:
    for label in sorted(df_resultados["mes"].unique()):
        subset = df_resultados[df_resultados["mes"] == label]
        carpeta_mes = os.path.join(BASE_DIR, label, 'PNG')
        if os.path.isdir(carpeta_mes):
            pngs = len(glob.glob(os.path.join(carpeta_mes, '*.png')))
            despejadas = len(subset[subset['nubes_%'] < 10])
            nubladas = len(subset[(subset['nubes_%'] >= 10) & (subset['nubes_%'] < 100)])
            sin_dato = len(subset[subset['nubes_%'] == -1])
            print(f"  {label}: {pngs} PNGs | {len(subset)} escenas")
        else:
            print(f"  {label}: {len(subset)} escenas (sin exportar)")
    print(f"\n  TOTAL: {len(df_resultados)} escenas encontradas")
print("=" * 60)